In [25]:
from pathlib import Path

import pyvista as pv
from la_construction_config import parameterization, workflow

from ulac.common import mesh_utils, serialization, visualization
from ulac.construction import check, constructor

In [26]:
patient_id = "07"
data_dir = Path(f"../data/patient_{patient_id}")
input_mesh_path = data_dir / "mesh_with_fibers_tags.vtu"
uac_submesh_path = data_dir / "uac_submesh_data.msgpack"
anatomical_tags = {"MV": 0, "LAA": 1, "LIPV": 2, "LSPV": 3, "RIPV": 4, "RSPV": 5}

mesh = pv.read(input_mesh_path)
mesh = mesh_utils.convert_unstructured_to_polydata_mesh(mesh)

In [27]:
ulac_construction_settings = constructor.UACConstructorSettings(
    mesh=mesh,
    feature_tags=anatomical_tags,
    path_config=parameterization.path_configs,
    marker_config=parameterization.marker_configs,
    parameterization_config=parameterization.parameterization_configs,
    uac_config=parameterization.uac_configs,
    submesh_config=parameterization.submesh_configs,
    segmentation_workflow=workflow.workflow,
)
ulac_constructor = constructor.UACConstructor(ulac_construction_settings)
ulac_constructor.construct_segmentation()
ulac_constructor.construct_uacs()

Starting Segmentation


Step 1/10: boundary_extraction
------------------------------
Extracting Feature: ('LIPV', 'inner')
Extracting Feature: ('LIPV', 'outer')
Extracting Feature: ('LSPV', 'inner')
Extracting Feature: ('LSPV', 'outer')
Extracting Feature: ('RSPV', 'inner')
Extracting Feature: ('RSPV', 'outer')
Extracting Feature: ('RIPV', 'inner')
Extracting Feature: ('RIPV', 'outer')
Extracting Feature: ('LAA',)
Extracting Feature: ('MV',)
 
Step 2/10: path_construction_roof_anchors
-----------------------------------------
Constructing Shortest Path: ('roof', 'LIPV_LSPV')
Constructing Shortest Path: ('roof', 'LSPV_RSPV')
Constructing Shortest Path: ('roof', 'RSPV_RIPV')
Constructing Shortest Path: ('roof', 'RIPV_LIPV')
Constructing Shortest Path: ('anchor', 'LIPV_LAA')
Constructing Shortest Path: ('anchor', 'LAA_MV')
Constructing Shortest Path: ('anchor', 'LSPV_MV')
Constructing Shortest Path: ('anchor', 'RSPV_MV')
Constructing Shortest Path: ('anchor', 'RIPV_MV')
 
Step 3/10: mark

In [28]:
visualization.plot_mesh_with_segmentation(
    ulac_constructor.data.marker_data,
    ulac_constructor.data.uac_path_data,
    mesh,
)
visualization.plot_uac_mesh(ulac_constructor.data.uac_submesh_data)
serialization.save_uac_submeshdata(uac_submesh_path, ulac_constructor.data.uac_submesh_data)
check.check_uac_mesh_validity(ulac_constructor.data.uac_submesh_data)

Widget(value='<iframe src="http://localhost:37597/index.html?ui=P_0x7f5f74ce5950_12&reconnect=auto" class="pyv…

Widget(value='<iframe src="http://localhost:37597/index.html?ui=P_0x7f5f74ce5590_13&reconnect=auto" class="pyv…

True